In [70]:
import torch.optim as optim
import pickle
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torch.cuda.amp import autocast, GradScaler
import wandb
import os
import numpy as np
import matplotlib.pyplot as plt
import torch._utils_internal
# from ConvolutionalAutoencoder import ConvolutionalAutoencoder

In [71]:
class ConvolutionalAutoencoder(nn.Module):
    def __init__(self, batch_size, latent_size=(8, 6)):
        super(ConvolutionalAutoencoder, self).__init__()

        #self.batch_size = batch_size
        self.latent_size = latent_size

        # Encoder layers
        self.encoder = nn.Sequential(
            nn.Conv1d(3, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Conv1d(16, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Conv1d(16, 8, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
            nn.Flatten(),
            nn.Linear(in_features=8*15, out_features=8*6)
        )

        # Decoder layers
        self.decoder = nn.Sequential(
            nn.Linear(in_features=8 * 6, out_features=8 * 16),  # Input: (batch_size, 8*6), Output: (batch_size, 8*16)
            nn.Unflatten(1, (8, 16)),  # Input: (batch_size, 8*16), Output: (batch_size, 8, 16)
            nn.ConvTranspose1d(8, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            # Input: (batch_size, 8, 16), Output: (batch_size, 16, 33)
            nn.ReLU(),
            nn.ConvTranspose1d(16, 16, kernel_size=3, stride=2, padding=1),
            # Input: (batch_size, 16, 33), Output: (batch_size, 16, 65)
            nn.ReLU(),
            nn.ConvTranspose1d(16, 16, kernel_size=3, stride=2, padding=1),
            # Input: (batch_size, 16, 65), Output: (batch_size, 16, 129)
            nn.ReLU(),
            nn.Conv1d(16, 3, kernel_size=5, stride=1, padding=2),
            # Input: (batch_size, 16, 129), Output: (batch_size, 3, 125)
            nn.ReLU()
        )

        # Reconstruction loss
        self.criterion = nn.MSELoss()

    def forward(self, x):
        print("here")
        encoded = self.encoder(x)
        print(encoded)
        encoded = encoded.view(x.size(0), *self.latent_size)  # Reshape the tensor to desired latent size
        print(encoded)

        assert encoded.size()[
               1:] == self.latent_size, f"Encoder output size {encoded.size()[1:]} does not match the specified latent size {self.latent_size}."

        decoded = self.decoder(encoded.view(x.size(0), -1))

        assert decoded.size()[1:] == (
        3, 125), f"Decoder output size {decoded.size()[1:]} does not match the specified output size (3, 125)."

        return decoded, encoded

In [2]:
class CustomDataset(torch.utils.data.Dataset):
    # Dataset class for loading and pre-processing data
    def __init__(self, data):
        self.data = data

    def __len__(self):
        # Returns the total number of samples
        return len(self.data)

    def __getitem__(self, idx):
        # Retrieves a sample at the given index idx
        sample = self.data[idx]
        sample = sample.view(3, 125)  # Reshapes the sample to match the input size of the model
        return sample

In [3]:
# data = pickle.load(open("_data_train_autoencoder_flat.pickle", "rb"))

In [52]:
# path = torch._utils_internal.resolve_library_path(path)
data = CustomDataset(pickle.load(open("_data_train_autoencoder_flat.pickle", "rb")))

In [53]:
# data_loader = torch.utils.data.DataLoader(data)

In [54]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps')
device

device(type='cuda')

In [72]:
model = ConvolutionalAutoencoder(batch_size=1).to(device)

In [73]:
model.load_state_dict(torch.load("saved_models/checkpoint_300.pth")["model_state_dict"])

<All keys matched successfully>

https://pytorch.org/tutorials/recipes/recipes/save_load_across_devices.html


In [74]:
print(model.encoder)

Sequential(
  (0): Conv1d(3, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (1): ReLU()
  (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv1d(16, 16, kernel_size=(3,), stride=(1,), padding=(1,))
  (4): ReLU()
  (5): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv1d(16, 8, kernel_size=(3,), stride=(1,), padding=(1,))
  (7): ReLU()
  (8): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (9): Flatten(start_dim=1, end_dim=-1)
  (10): Linear(in_features=120, out_features=48, bias=True)
)


In [75]:
# len(data_loader[0])

In [78]:
# Set the model to evaluation mode
model.eval()

# Encode your data using the pretrained encoder
input_data = data[0].float().to(device)
# encoded_data = model.encoder(input_data)
reconstruction, encoded = model(input_data)

# Use the encoded data for another algorithm
# Example: Print the encoded data
# print(encoded)

here


RuntimeError: mat1 and mat2 shapes cannot be multiplied (8x15 and 120x48)

In [21]:
data[0].float().to(device).shape

torch.Size([3, 125])

In [18]:
input_data = torch.tensor(data[0].float().to(device))
input_data

/tmp/ipykernel_184134/910591254.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_data = torch.tensor(data[0].float().to(device))


tensor([[ 0.4099, -0.0300,  0.1200,  0.4299, -0.0100,  0.1200,  0.4500,  0.0000,
          0.1100,  0.4600,  0.0100,  0.1000,  0.4700,  0.0300,  0.0800,  0.3799,
         -0.0200,  0.1200,  0.4099,  0.0000,  0.1300,  0.4299,  0.0200,  0.1200,
          0.4500,  0.0300,  0.1000,  0.4600,  0.0400,  0.0800,  0.3601, -0.0100,
          0.1200,  0.3899,  0.0200,  0.1200,  0.4199,  0.0300,  0.1200,  0.4399,
          0.0500,  0.1000,  0.4600,  0.0600,  0.0800,  0.3401,  0.0000,  0.1100,
          0.3701,  0.0300,  0.1200,  0.3999,  0.0500,  0.1100,  0.4299,  0.0700,
          0.1000,  0.4500,  0.0800,  0.0700,  0.3301,  0.0100,  0.1000,  0.3501,
          0.0300,  0.1100,  0.3799,  0.0500,  0.1100,  0.4099,  0.0800,  0.1000,
          0.4299,  0.1000,  0.0700,  0.3999, -0.0500,  0.1400,  0.4199, -0.0300,
          0.1300,  0.4399, -0.0100,  0.1300,  0.4600,  0.0100,  0.1100,  0.4700,
          0.0200,  0.0900,  0.3701, -0.0400,  0.1400,  0.3999, -0.0100,  0.1400,
          0.4299,  0.0100,  